# Bronze multi-lane ingest

Load and validate the six synthetic source lanes without mixing their grains, then publish one Delta table per lane plus an ingest manifest.

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd

DATA_PRODUCT_PATH = Path("shared/integrated-test-data/projections/star-schema")
DATA_ROOT_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_PRODUCT_PATH,
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / DATA_PRODUCT_PATH,
]

data_root = next(
    (
        candidate
        for candidate in DATA_ROOT_CANDIDATES
        if (candidate / "demo05_support.py").exists()
        and (candidate / "scenario_topology.csv").exists()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError(
        "Shared star-schema projection was not found in Lakehouse Files or the local repository."
    )

if str(data_root) not in sys.path:
    sys.path.insert(0, str(data_root))
import demo05_support as demo

importlib.reload(demo)
sources = demo.load_sources(data_root)
source_checks = demo.validate_source_frames(sources)
failed_checks = source_checks.loc[~source_checks["passed"]]
if not failed_checks.empty:
    raise ValueError(f"Source contract validation failed:\n{failed_checks.to_string(index=False)}")

bronze_tables = {
    f"bronze_{lane_name}": frame for lane_name, frame in sources.items()
}
bronze_tables["bronze_ingest_manifest"] = pd.DataFrame(
    [
        {
            "source_lane": lane_name,
            "source_file": demo.SOURCE_FILES[lane_name],
            "row_count": len(frame),
            "classification": demo.SYNTHETIC_CLASSIFICATION,
        }
        for lane_name, frame in sources.items()
    ]
)

spark_session = globals().get("spark")
if spark_session is None:
    print("Spark is unavailable; validated Bronze tables without publishing Delta tables.")
else:
    for table_name, frame in bronze_tables.items():
        (
            spark_session.createDataFrame(demo.spark_compatible_frame(frame))
            .write.mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )

print({table_name: len(frame) for table_name, frame in bronze_tables.items()})
source_checks